In [1]:
!pip install triton

In [2]:
import torch
import triton
import triton.language as tl

In [3]:
def layernorm_forward_torch(x, weight, bias, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    rstd = 1.0 / torch.sqrt(var + eps)
    x_hat = (x - mean) * rstd
    return x_hat * weight + bias

## triton kernels

In [4]:
_AUTOTUNE_CONFIGS = [triton.Config({}, num_warps=w) for w in (1, 2, 4, 8, 16, 32)]


@triton.autotune(configs=_AUTOTUNE_CONFIGS, key=['N'])
@triton.jit
def _layernorm_fwd_kernel(
    X_ptr, Y_ptr, W_ptr, B_ptr, Mean_ptr, Rstd_ptr,
    stride_xm, stride_ym,
    N, eps,
    BLOCK_N: tl.constexpr,
):
    pid = tl.program_id(0)
    offs = tl.arange(0, BLOCK_N)
    mask = offs < N

    x = tl.load(X_ptr + pid * stride_xm + offs, mask=mask, other=0.0).to(tl.float32)
    mean = tl.sum(x, axis=0) / N
    x_centered = tl.where(mask, x - mean, 0.0)
    var = tl.sum(x_centered * x_centered, axis=0) / N
    rstd = 1.0 / tl.sqrt(var + eps)

    tl.store(Mean_ptr + pid, mean)
    tl.store(Rstd_ptr + pid, rstd)

    w = tl.load(W_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    b = tl.load(B_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    y = x_centered * rstd * w + b
    tl.store(Y_ptr + pid * stride_ym + offs, y, mask=mask)

In [5]:
# reset_to_zero обязателен: autotune прогоняет кернел много раз на одном буфере,
# а dw/db копятся через atomic_add — без зануления вклад складывается десятки раз
@triton.autotune(configs=_AUTOTUNE_CONFIGS, key=['N'], reset_to_zero=['DW_ptr', 'DB_ptr'])
@triton.jit
def _layernorm_bwd_kernel(
    X_ptr, W_ptr, Mean_ptr, Rstd_ptr,
    DY_ptr, DX_ptr, DW_ptr, DB_ptr,
    stride_xm, stride_dym, stride_dxm,
    N,
    BLOCK_N: tl.constexpr,
):
    pid = tl.program_id(0)
    offs = tl.arange(0, BLOCK_N)
    mask = offs < N

    x = tl.load(X_ptr + pid * stride_xm + offs, mask=mask, other=0.0).to(tl.float32)
    dy = tl.load(DY_ptr + pid * stride_dym + offs, mask=mask, other=0.0).to(tl.float32)
    w = tl.load(W_ptr + offs, mask=mask, other=0.0).to(tl.float32)
    mean = tl.load(Mean_ptr + pid)
    rstd = tl.load(Rstd_ptr + pid)

    x_hat = (x - mean) * rstd
    dx_hat = dy * w

    sum_dx_hat = tl.sum(tl.where(mask, dx_hat, 0.0), axis=0)
    sum_dx_hat_x_hat = tl.sum(tl.where(mask, dx_hat * x_hat, 0.0), axis=0)

    # dx_i = rstd/N * (N*dx_hat_i - sum(dx_hat) - x_hat_i * sum(dx_hat * x_hat))
    dx = rstd * (dx_hat - (sum_dx_hat + x_hat * sum_dx_hat_x_hat) / N)
    tl.store(DX_ptr + pid * stride_dxm + offs, dx, mask=mask)

    # dw и db копятся по всем строкам M, поэтому атомарно докидываем вклад этой строки
    tl.atomic_add(DW_ptr + offs, dy * x_hat, mask=mask)
    tl.atomic_add(DB_ptr + offs, dy, mask=mask)

## autograd.Function-обёртка

In [6]:
class LayerNormTriton(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, weight, bias, eps=1e-5):
        x_shape = x.shape
        N = x_shape[-1]
        x_flat = x.reshape(-1, N).contiguous()
        M = x_flat.shape[0]

        y = torch.empty_like(x_flat)
        mean = torch.empty(M, device=x.device, dtype=torch.float32)
        rstd = torch.empty(M, device=x.device, dtype=torch.float32)
        BLOCK_N = triton.next_power_of_2(N)

        _layernorm_fwd_kernel[(M,)](
            x_flat, y, weight, bias, mean, rstd,
            x_flat.stride(0), y.stride(0),
            N, eps,
            BLOCK_N=BLOCK_N,
        )
        ctx.save_for_backward(x_flat, weight, mean, rstd)
        ctx.x_shape = x_shape
        ctx.BLOCK_N = BLOCK_N
        return y.reshape(x_shape)

    @staticmethod
    def backward(ctx, dy):
        x_flat, weight, mean, rstd = ctx.saved_tensors
        x_shape = ctx.x_shape
        BLOCK_N = ctx.BLOCK_N
        M, N = x_flat.shape

        dy_flat = dy.reshape(-1, N).contiguous()
        dx = torch.empty_like(x_flat)
        # копим в fp32, атомики на fp16 есть не везде и точность хуже
        dw = torch.zeros(N, device=weight.device, dtype=torch.float32)
        db = torch.zeros(N, device=weight.device, dtype=torch.float32)

        _layernorm_bwd_kernel[(M,)](
            x_flat, weight, mean, rstd,
            dy_flat, dx, dw, db,
            x_flat.stride(0), dy_flat.stride(0), dx.stride(0),
            N,
            BLOCK_N=BLOCK_N,
        )
        return dx.reshape(x_shape), dw.to(weight.dtype), db.to(weight.dtype), None


def layernorm_triton(x, weight, bias, eps=1e-5):
    return LayerNormTriton.apply(x, weight, bias, eps)

## проверка

In [7]:
def test_correctness():
    torch.manual_seed(0)
    M, N = 1024, 512
    x = torch.randn(M, N, device='cuda', dtype=torch.float32, requires_grad=True)
    weight = torch.randn(N, device='cuda', dtype=torch.float32, requires_grad=True)
    bias = torch.randn(N, device='cuda', dtype=torch.float32, requires_grad=True)

    x_ref = x.detach().clone().requires_grad_()
    w_ref = weight.detach().clone().requires_grad_()
    b_ref = bias.detach().clone().requires_grad_()

    y_ref = torch.nn.functional.layer_norm(x_ref, (N,), w_ref, b_ref, eps=1e-5)
    y_triton = layernorm_triton(x, weight, bias, eps=1e-5)
    torch.testing.assert_close(y_triton, y_ref, atol=1e-4, rtol=1e-4)

    dy = torch.randn_like(y_ref)
    y_ref.backward(dy)
    y_triton.backward(dy)

    torch.testing.assert_close(x.grad, x_ref.grad, atol=1e-3, rtol=1e-3)
    torch.testing.assert_close(weight.grad, w_ref.grad, atol=1e-3, rtol=1e-3)
    torch.testing.assert_close(bias.grad, b_ref.grad, atol=1e-3, rtol=1e-3)
    print('ok, forward + backward совпадают с torch.nn.functional.layer_norm')

In [8]:
test_correctness()

ok, forward + backward совпадают с torch.nn.functional.layer_norm


## бенчмарк против torch

In [9]:
def bench():
    print(f"{'M':>8}{'N':>8}{'mode':>10}{'triton GB/s':>16}{'torch GB/s':>16}{'speedup':>10}")
    for N in (512, 1024, 2048, 4096, 8192):
        for mode in ('forward', 'backward'):
            M = 4096
            x = torch.randn(M, N, device='cuda', dtype=torch.float32, requires_grad=(mode == 'backward'))
            weight = torch.randn(N, device='cuda', dtype=torch.float32, requires_grad=(mode == 'backward'))
            bias = torch.randn(N, device='cuda', dtype=torch.float32, requires_grad=(mode == 'backward'))

            def make_run(fn):
                if mode == 'forward':
                    return lambda: fn(x, weight, bias)
                def run():
                    y = fn(x, weight, bias)
                    dy = torch.randn_like(y)
                    y.backward(dy, retain_graph=True)
                return run

            triton_run = make_run(layernorm_triton)
            torch_run = make_run(lambda a, b, c: torch.nn.functional.layer_norm(a, (N,), b, c))

            ms_t = triton.testing.do_bench(triton_run)
            ms_p = triton.testing.do_bench(torch_run)

            # forward: читаем x, пишем y. backward: читаем x, dy, пишем dx + 2 атомика, грубо 4x
            bytes_total = x.numel() * x.element_size() * (2 if mode == 'forward' else 4)
            gb_t = bytes_total / (ms_t * 1e6)
            gb_p = bytes_total / (ms_p * 1e6)
            print(f"{M:>8}{N:>8}{mode:>10}{gb_t:>16.1f}{gb_p:>16.1f}{ms_p/ms_t:>9.2f}x")

In [10]:
bench()

       M       N      mode     triton GB/s      torch GB/s   speedup
    4096     512   forward           218.8           224.4     0.98x
    4096     512  backward            86.7            83.9     1.03x
    4096    1024   forward           236.3           218.2     1.08x
    4096    1024  backward            81.2            79.0     1.03x
    4096    2048   forward           239.9           176.3     1.36x
    4096    2048  backward            88.1            70.7     1.25x
    4096    4096   forward           243.4           163.3     1.49x
    4096    4096  backward            84.7            69.3     1.22x
    4096    8192   forward           238.5           162.9     1.46x
    4096    8192  backward            76.9            69.2     1.11x
